# Trash Sorting Robot — Transfer Learning dengan MobileNetV2

Notebook ini melatih model klasifikasi gambar sampah menggunakan transfer learning. Dataset yang dipakai: TrashNet (enam kelas). Jalankan sel secara berurutan di Google Colab.

**Penting:** metrik dan model baru tersedia setelah notebook dijalankan. Jangan melaporkan hasil yang belum diukur.


In [ ]:
!pip -q install tensorflow matplotlib scikit-learn


## 1. Unduh dan siapkan dataset
Unduh dataset dari repositori sumber: https://github.com/garythung/trashnet . Unggah ZIP dataset ke Google Colab, lalu sesuaikan `ZIP_PATH` di bawah. ZIP dapat memiliki folder pembungkus; kode mencoba menemukan direktori yang berisi subfolder kelas.


In [ ]:
import os, zipfile, glob, json, shutil, pathlib
ZIP_PATH = '/content/dataset-resized.zip'  # Ubah sesuai nama/lokasi file ZIP yang diunggah
EXTRACT_DIR = '/content/trashnet_data'
if not os.path.exists(ZIP_PATH):
    from google.colab import files
    uploaded = files.upload()
    ZIP_PATH = '/content/' + next(iter(uploaded))
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)
print('Ekstraksi selesai:', EXTRACT_DIR)
for root, dirs, files_ in os.walk(EXTRACT_DIR):
    if len(dirs) >= 2:
        image_count = sum(1 for f in files_ if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')))
        if image_count or any(os.path.isdir(os.path.join(root, d)) for d in dirs):
            print('Folder kandidat:', root, '| subfolder:', dirs[:8], '| gambar langsung:', image_count)


In [ ]:
import tensorflow as tf
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
def find_dataset_root(start):
    candidates = []
    for root, dirs, files_ in os.walk(start):
        class_dirs = [d for d in dirs if not d.startswith('.')]
        if len(class_dirs) >= 2:
            direct_images = sum(1 for f in files_ if f.lower().endswith(VALID_EXTS))
            if direct_images == 0:
                candidates.append((len(class_dirs), root))
    if not candidates:
        raise FileNotFoundError('Folder kelas tidak ditemukan. Periksa struktur ZIP dataset.')
    candidates.sort(reverse=True)
    return candidates[0][1]
DATASET_DIR = find_dataset_root(EXTRACT_DIR)
print('Dataset directory:', DATASET_DIR)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')
class_names = train_ds.class_names
print('Kelas:', class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


## 2. Bangun model transfer learning
MobileNetV2 menggunakan bobot ImageNet sebagai ekstraktor fitur. Lapisan dasar dibekukan pada tahap awal, lalu ditambahkan kepala klasifikasi sesuai jumlah kelas dataset.


In [ ]:
from tensorflow.keras import layers, Model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False
inputs = layers.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
OUT_DIR = '/content/TrashSortingRobot_output'
os.makedirs(OUT_DIR + '/models', exist_ok=True)
os.makedirs(OUT_DIR + '/results', exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(OUT_DIR + '/models/trash_classifier.keras', monitor='val_loss', save_best_only=True)
]
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)
model.save(OUT_DIR + '/models/trash_classifier.keras')
with open(OUT_DIR + '/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)


## 3. Evaluasi dan simpan grafik


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_acc:.4f}')
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('MobileNetV2 Transfer Learning')
plt.legend(); plt.grid(True)
plt.savefig(OUT_DIR + '/results/training_accuracy.png', bbox_inches='tight', dpi=150)
plt.show()
y_true, y_pred = [], []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(probs.argmax(axis=1).tolist())
report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
print(report)
with open(OUT_DIR + '/results/classification_report.txt', 'w') as f:
    f.write(report)
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8,8)); disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title('Confusion Matrix'); plt.tight_layout()
plt.savefig(OUT_DIR + '/results/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
with open(OUT_DIR + '/results/metrics.json', 'w') as f:
    json.dump({'validation_loss': float(val_loss), 'validation_accuracy': float(val_acc), 'classes': class_names}, f, indent=2)


## 4. Uji satu gambar dan unduh hasil
Pilih gambar sampah untuk melihat prediksi model. Untuk penggunaan robot sebenarnya, model perlu diuji dengan foto dari kamera robot dan kondisi pencahayaan yang berbeda.


In [ ]:
from PIL import Image
import numpy as np
from google.colab import files
uploaded_test = files.upload()
test_path = '/content/' + next(iter(uploaded_test))
img = tf.keras.utils.load_img(test_path, target_size=IMG_SIZE)
arr = tf.keras.utils.img_to_array(img)
probs = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
idx = int(np.argmax(probs))
print('Prediksi:', class_names[idx])
print(f'Confidence model: {probs[idx]:.2%} (bukan jaminan prediksi benar)')
display(img)


In [ ]:
from google.colab import files
shutil.make_archive('/content/TrashSortingRobot_results', 'zip', OUT_DIR)
files.download('/content/TrashSortingRobot_results.zip')
